---
## 1. Imports & Configuration

In [1]:
import os
import math
import random
from collections import Counter

import pyarrow.parquet as pq
from tqdm import tqdm

# ── Paths ──────────────────────────────────────────────────────────────────
PARQUET_DIR = os.path.join("..", "Assignment_1", "data", "output")
FIRST_SHARD = os.path.join(PARQUET_DIR, "indiccorp_te_tokenized_part-00001.parquet")

# ── Parameters ─────────────────────────────────────────────────────────────
NUM_SENTENCES = 1_000_000
DEV_SIZE      = 1_000
TEST_SIZE     = 1_000
TRAIN_SIZE    = NUM_SENTENCES - DEV_SIZE - TEST_SIZE   # 998,000
SEED          = 42

# ── Add-K parameter ───────────────────────────────────────────────────────
K = 0.3

# Special tokens
SOS = "<s>"
EOS = "</s>"
UNK = "<UNK>"

print(f"Add-K Smoothing with K = {K}")
print(f"Will load {NUM_SENTENCES:,} sentences from:\n  {os.path.abspath(FIRST_SHARD)}")
print(f"Split: train={TRAIN_SIZE:,}  dev={DEV_SIZE:,}  test={TEST_SIZE:,}")

Add-K Smoothing with K = 0.3
Will load 1,000,000 sentences from:
  d:\Assignments_sem_5\NLP\Assignment_1\data\output\indiccorp_te_tokenized_part-00001.parquet
Split: train=998,000  dev=1,000  test=1,000


---
## 2. Load Data & Train / Dev / Test Split

Same loading and splitting procedure as Assignment 4 (same seed) for consistency.

In [2]:
# Load parquet data
table = pq.read_table(FIRST_SHARD, columns=["tokenized_sentence"])
raw_sentences = table.column("tokenized_sentence").to_pylist()

# Tokenize
sentences = [s.split() for s in raw_sentences[:NUM_SENTENCES]]

# Shuffle with same seed as Assignment 4 for identical splits
random.seed(SEED)
random.shuffle(sentences)

train_sentences = sentences[:TRAIN_SIZE]
dev_sentences   = sentences[TRAIN_SIZE : TRAIN_SIZE + DEV_SIZE]
test_sentences  = sentences[TRAIN_SIZE + DEV_SIZE : TRAIN_SIZE + DEV_SIZE + TEST_SIZE]

print(f"Training set   : {len(train_sentences):>10,} sentences")
print(f"Development set: {len(dev_sentences):>10,} sentences")
print(f"Test set       : {len(test_sentences):>10,} sentences")

del raw_sentences, table, sentences

Training set   :    998,000 sentences
Development set:      1,000 sentences
Test set       :      1,000 sentences


---
## 3. Build Vocabulary & N-gram Counts

In [3]:
def pad_sentence(tokens, n):
    """Pad a sentence with (n-1) <s> tokens at the start and one </s> at the end."""
    return [SOS] * (n - 1) + tokens + [EOS]


def get_ngrams(tokens, n):
    """Extract all n-grams from a token list."""
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

In [4]:
# ── Build vocabulary ─────────────────────────────────────────────────────
vocab = set()
for sent in tqdm(train_sentences, desc="Building vocabulary"):
    vocab.update(sent)

vocab.add(SOS)
vocab.add(EOS)
vocab.add(UNK)

V = len(vocab)
print(f"Vocabulary size (V): {V:,}")

Building vocabulary: 100%|██████████| 998000/998000 [00:02<00:00, 372207.06it/s]

Vocabulary size (V): 670,753


In [5]:
# ── Count n-grams ────────────────────────────────────────────────────────
unigram_counts = Counter()

bigram_counts   = Counter()
bigram_context  = Counter()

trigram_counts   = Counter()
trigram_context  = Counter()

quadrigram_counts   = Counter()
quadrigram_context  = Counter()

for sent in tqdm(train_sentences, desc="Counting n-grams"):
    # Unigrams
    padded_1 = pad_sentence(sent, 1)
    for gram in get_ngrams(padded_1, 1):
        unigram_counts[gram] += 1
    
    # Bigrams
    padded_2 = pad_sentence(sent, 2)
    for gram in get_ngrams(padded_2, 2):
        bigram_counts[gram] += 1
        bigram_context[gram[:1]] += 1
    
    # Trigrams
    padded_3 = pad_sentence(sent, 3)
    for gram in get_ngrams(padded_3, 3):
        trigram_counts[gram] += 1
        trigram_context[gram[:2]] += 1
    
    # Quadrigrams
    padded_4 = pad_sentence(sent, 4)
    for gram in get_ngrams(padded_4, 4):
        quadrigram_counts[gram] += 1
        quadrigram_context[gram[:3]] += 1

N_unigrams = sum(unigram_counts.values())

print(f"\n{'Model':<12} {'Unique n-grams':>16} {'Total count':>14}")
print("-" * 46)
print(f"{'Unigram':<12} {len(unigram_counts):>16,} {N_unigrams:>14,}")
print(f"{'Bigram':<12} {len(bigram_counts):>16,} {sum(bigram_counts.values()):>14,}")
print(f"{'Trigram':<12} {len(trigram_counts):>16,} {sum(trigram_counts.values()):>14,}")
print(f"{'Quadrigram':<12} {len(quadrigram_counts):>16,} {sum(quadrigram_counts.values()):>14,}")

Counting n-grams: 100%|██████████| 998000/998000 [01:10<00:00, 14064.01it/s]



Model          Unique n-grams    Total count
----------------------------------------------
Unigram               670,751     11,439,087
Bigram              4,797,059     11,439,087
Trigram             7,831,299     11,439,087
Quadrigram          9,045,911     11,439,087


In [6]:
def unigram_prob_addk(word):
    """Add-K smoothed unigram: P(w) = (C(w) + k) / (N + k*V)"""
    count = unigram_counts.get((word,), 0)
    return (count + K) / (N_unigrams + K * V)


def sentence_prob_unigram_addk(tokens):
    """Compute log probability of a sentence under the Add-K unigram model."""
    padded = pad_sentence(tokens, 1)
    log_prob = 0.0
    n_tokens = 0
    for w in padded:
        p = unigram_prob_addk(w)
        log_prob += math.log2(p)
        n_tokens += 1
    return log_prob, n_tokens


# ── Verify: known token vs. unknown token ────────────────────────────────
sample_token = list(unigram_counts.keys())[0][0]
print(f"K = {K}")
print(f"P_add-k('{sample_token}')    = {unigram_prob_addk(sample_token):.10f}")
print(f"P_add-k('<UNK>')             = {unigram_prob_addk(UNK):.10f}")
print(f"P_add-k('NEVER_SEEN_XYZ')    = {unigram_prob_addk('NEVER_SEEN_XYZ'):.10f}")
print(f"\nAll probabilities > 0 (no zero-probability issue)")

K = 0.3
P_add-k('పెట్రోల్')    = 0.0000849032
P_add-k('<UNK>')             = 0.0000000258
P_add-k('NEVER_SEEN_XYZ')    = 0.0000000258

All probabilities > 0 (no zero-probability issue)


In [7]:
def bigram_prob_addk(w_prev, w_curr):
    """Add-K smoothed bigram: P(w_curr | w_prev) = (C(w_prev, w_curr) + k) / (C(w_prev) + k*V)"""
    numerator   = bigram_counts.get((w_prev, w_curr), 0) + K
    denominator = bigram_context.get((w_prev,), 0) + K * V
    return numerator / denominator


def sentence_prob_bigram_addk(tokens):
    """Compute log probability under the Add-K bigram model."""
    padded = pad_sentence(tokens, 2)
    log_prob = 0.0
    n_tokens = 0
    for i in range(1, len(padded)):
        p = bigram_prob_addk(padded[i-1], padded[i])
        log_prob += math.log2(p)
        n_tokens += 1
    return log_prob, n_tokens


# ── Show sample conditional probabilities ────────────────────────────────
print(f"Sample Add-K (K={K}) bigram probabilities:\n")
top_bigrams = bigram_counts.most_common(5)
for (w1, w2), count in top_bigrams:
    p_addk = bigram_prob_addk(w1, w2)
    p_raw  = count / bigram_context.get((w1,), 1)
    print(f"  P({w2} | {w1})  MLE={p_raw:.6f}  Add-K={p_addk:.6f}")

Sample Add-K (K=0.3) bigram probabilities:

  P(</s> | .)  MLE=0.991595  Add-K=0.810927
  P(ఈ | <s>)  MLE=0.051941  Add-K=0.043226
  P(. | ఉంది)  MLE=0.866796  Add-K=0.110421
  P(. | చేశారు)  MLE=0.954310  Add-K=0.095723
  P(</s> | ?)  MLE=0.913245  Add-K=0.086908


In [8]:
def trigram_prob_addk(w_2, w_1, w_0):
    """Add-K smoothed trigram: P(w_0 | w_2, w_1) = (C(w_2, w_1, w_0) + k) / (C(w_2, w_1) + k*V)"""
    numerator   = trigram_counts.get((w_2, w_1, w_0), 0) + K
    denominator = trigram_context.get((w_2, w_1), 0) + K * V
    return numerator / denominator


def sentence_prob_trigram_addk(tokens):
    """Compute log probability under the Add-K trigram model."""
    padded = pad_sentence(tokens, 3)
    log_prob = 0.0
    n_tokens = 0
    for i in range(2, len(padded)):
        p = trigram_prob_addk(padded[i-2], padded[i-1], padded[i])
        log_prob += math.log2(p)
        n_tokens += 1
    return log_prob, n_tokens


# ── Show sample conditional probabilities ────────────────────────────────
print(f"Sample Add-K (K={K}) trigram probabilities:\n")
top_trigrams = trigram_counts.most_common(5)
for (w1, w2, w3), count in top_trigrams:
    p_addk = trigram_prob_addk(w1, w2, w3)
    p_raw  = count / trigram_context.get((w1, w2), 1)
    print(f"  P({w3} | {w1}, {w2})  MLE={p_raw:.6f}  Add-K={p_addk:.6f}")

Sample Add-K (K=0.3) trigram probabilities:

  P(ఈ | <s>, <s>)  MLE=0.051941  Add-K=0.043226
  P(</s> | ఉంది, .)  MLE=0.998311  Add-K=0.112137
  P(</s> | చేశారు, .)  MLE=0.999113  Add-K=0.096078
  P(</s> | ఉన్నాయి, .)  MLE=0.999169  Add-K=0.087298
  P(అయితే | <s>, <s>)  MLE=0.018929  Add-K=0.015753


In [9]:
def quadrigram_prob_addk(w_3, w_2, w_1, w_0):
    """Add-K smoothed quadrigram: P(w_0 | w_3, w_2, w_1) = (C(w_3, w_2, w_1, w_0) + k) / (C(w_3, w_2, w_1) + k*V)"""
    numerator   = quadrigram_counts.get((w_3, w_2, w_1, w_0), 0) + K
    denominator = quadrigram_context.get((w_3, w_2, w_1), 0) + K * V
    return numerator / denominator


def sentence_prob_quadrigram_addk(tokens):
    """Compute log probability under the Add-K quadrigram model."""
    padded = pad_sentence(tokens, 4)
    log_prob = 0.0
    n_tokens = 0
    for i in range(3, len(padded)):
        p = quadrigram_prob_addk(padded[i-3], padded[i-2], padded[i-1], padded[i])
        log_prob += math.log2(p)
        n_tokens += 1
    return log_prob, n_tokens


# ── Show sample conditional probabilities ────────────────────────────────
print(f"Sample Add-K (K={K}) quadrigram probabilities:\n")
top_quadrigrams = quadrigram_counts.most_common(5)
for (w1, w2, w3, w4), count in top_quadrigrams:
    p_addk = quadrigram_prob_addk(w1, w2, w3, w4)
    p_raw  = count / quadrigram_context.get((w1, w2, w3), 1)
    print(f"  P({w4} | {w1}, {w2}, {w3})  MLE={p_raw:.6f}  Add-K={p_addk:.6f}")

Sample Add-K (K=0.3) quadrigram probabilities:

  P(ఈ | <s>, <s>, <s>)  MLE=0.051941  Add-K=0.043226
  P(అయితే | <s>, <s>, <s>)  MLE=0.018929  Add-K=0.015753
  P(ఆ | <s>, <s>, <s>)  MLE=0.014529  Add-K=0.012091
  P(దీంతో | <s>, <s>, <s>)  MLE=0.010470  Add-K=0.008713
  P(కానీ | <s>, <s>, <s>)  MLE=0.010245  Add-K=0.008527


---
## 9. Perplexity — Helper

In [10]:
def compute_perplexity(data_sentences, sentence_prob_fn):
    """
    Compute perplexity of a dataset under a given language model.
    
    PP(W) = 2^{ -1/N * sum_i log2 P(w_i | context) }
    
    With Add-K smoothing, no token has zero probability,
    so perplexity will always be finite.
    """
    total_log_prob = 0.0
    total_tokens   = 0
    
    for sent in data_sentences:
        lp, nt = sentence_prob_fn(sent)
        if math.isinf(lp):
            return float('inf')
        total_log_prob += lp
        total_tokens   += nt
    
    avg_log_prob = total_log_prob / total_tokens
    perplexity   = 2 ** (-avg_log_prob)
    return perplexity

---
## 10. Evaluation — Perplexity on Dev & Test Sets (Add-K Smoothed)

In [11]:
addk_models = {
    "Unigram":    sentence_prob_unigram_addk,
    "Bigram":     sentence_prob_bigram_addk,
    "Trigram":    sentence_prob_trigram_addk,
    "Quadrigram": sentence_prob_quadrigram_addk,
}

addk_results = {}

print(f"Perplexity with Add-K Smoothing (K = {K}):\n")
print(f"{'Model':<12} {'Dev PP':>15} {'Test PP':>15}")
print("-" * 45)

for name, prob_fn in addk_models.items():
    pp_dev  = compute_perplexity(dev_sentences, prob_fn)
    pp_test = compute_perplexity(test_sentences, prob_fn)
    addk_results[name] = {"dev": pp_dev, "test": pp_test}
    
    dev_str  = f"{pp_dev:,.2f}" if not math.isinf(pp_dev) else "inf"
    test_str = f"{pp_test:,.2f}" if not math.isinf(pp_test) else "inf"
    print(f"{name:<12} {dev_str:>15} {test_str:>15}")

Perplexity with Add-K Smoothing (K = 0.3):

Model                 Dev PP         Test PP
---------------------------------------------
Unigram             6,507.09        6,547.45
Bigram             17,685.77       18,637.74
Trigram           121,630.75      126,452.18
Quadrigram        228,002.44      236,956.11


---
## 11. Comparison: MLE vs. Add-1 (Laplace) vs. Add-K (K=0.3)

We also compute unsmoothed (MLE) and Laplace (Add-1) perplexity for a three-way comparison.

In [12]:
# ── MLE (unsmoothed) probability functions ───────────────────────────────

def sentence_prob_unigram_mle(tokens):
    padded = pad_sentence(tokens, 1)
    log_prob, n_tokens = 0.0, 0
    for w in padded:
        c = unigram_counts.get((w,), 0)
        log_prob += math.log2(c / N_unigrams) if c > 0 else float('-inf')
        n_tokens += 1
    return log_prob, n_tokens

def sentence_prob_bigram_mle(tokens):
    padded = pad_sentence(tokens, 2)
    log_prob, n_tokens = 0.0, 0
    for i in range(1, len(padded)):
        num = bigram_counts.get((padded[i-1], padded[i]), 0)
        den = bigram_context.get((padded[i-1],), 0)
        p = num / den if den > 0 else 0.0
        log_prob += math.log2(p) if p > 0 else float('-inf')
        n_tokens += 1
    return log_prob, n_tokens

def sentence_prob_trigram_mle(tokens):
    padded = pad_sentence(tokens, 3)
    log_prob, n_tokens = 0.0, 0
    for i in range(2, len(padded)):
        num = trigram_counts.get((padded[i-2], padded[i-1], padded[i]), 0)
        den = trigram_context.get((padded[i-2], padded[i-1]), 0)
        p = num / den if den > 0 else 0.0
        log_prob += math.log2(p) if p > 0 else float('-inf')
        n_tokens += 1
    return log_prob, n_tokens

def sentence_prob_quadrigram_mle(tokens):
    padded = pad_sentence(tokens, 4)
    log_prob, n_tokens = 0.0, 0
    for i in range(3, len(padded)):
        num = quadrigram_counts.get((padded[i-3], padded[i-2], padded[i-1], padded[i]), 0)
        den = quadrigram_context.get((padded[i-3], padded[i-2], padded[i-1]), 0)
        p = num / den if den > 0 else 0.0
        log_prob += math.log2(p) if p > 0 else float('-inf')
        n_tokens += 1
    return log_prob, n_tokens

In [13]:
# ── Laplace (Add-1) probability functions ────────────────────────────────

def sentence_prob_unigram_laplace(tokens):
    padded = pad_sentence(tokens, 1)
    log_prob, n_tokens = 0.0, 0
    for w in padded:
        c = unigram_counts.get((w,), 0)
        p = (c + 1) / (N_unigrams + V)
        log_prob += math.log2(p)
        n_tokens += 1
    return log_prob, n_tokens

def sentence_prob_bigram_laplace(tokens):
    padded = pad_sentence(tokens, 2)
    log_prob, n_tokens = 0.0, 0
    for i in range(1, len(padded)):
        num = bigram_counts.get((padded[i-1], padded[i]), 0) + 1
        den = bigram_context.get((padded[i-1],), 0) + V
        log_prob += math.log2(num / den)
        n_tokens += 1
    return log_prob, n_tokens

def sentence_prob_trigram_laplace(tokens):
    padded = pad_sentence(tokens, 3)
    log_prob, n_tokens = 0.0, 0
    for i in range(2, len(padded)):
        num = trigram_counts.get((padded[i-2], padded[i-1], padded[i]), 0) + 1
        den = trigram_context.get((padded[i-2], padded[i-1]), 0) + V
        log_prob += math.log2(num / den)
        n_tokens += 1
    return log_prob, n_tokens

def sentence_prob_quadrigram_laplace(tokens):
    padded = pad_sentence(tokens, 4)
    log_prob, n_tokens = 0.0, 0
    for i in range(3, len(padded)):
        num = quadrigram_counts.get((padded[i-3], padded[i-2], padded[i-1], padded[i]), 0) + 1
        den = quadrigram_context.get((padded[i-3], padded[i-2], padded[i-1]), 0) + V
        log_prob += math.log2(num / den)
        n_tokens += 1
    return log_prob, n_tokens

In [14]:
# ── Compute all perplexities ─────────────────────────────────────────────
model_names = ["Unigram", "Bigram", "Trigram", "Quadrigram"]

mle_fns     = [sentence_prob_unigram_mle, sentence_prob_bigram_mle,
               sentence_prob_trigram_mle, sentence_prob_quadrigram_mle]
laplace_fns = [sentence_prob_unigram_laplace, sentence_prob_bigram_laplace,
               sentence_prob_trigram_laplace, sentence_prob_quadrigram_laplace]
addk_fns    = [sentence_prob_unigram_addk, sentence_prob_bigram_addk,
               sentence_prob_trigram_addk, sentence_prob_quadrigram_addk]

all_results = {}

print("Computing perplexity for MLE, Laplace (k=1), and Add-K (k=0.3)...\n")

for name, mle_fn, lap_fn, addk_fn in zip(model_names, mle_fns, laplace_fns, addk_fns):
    all_results[name] = {
        "mle_dev":     compute_perplexity(dev_sentences, mle_fn),
        "mle_test":    compute_perplexity(test_sentences, mle_fn),
        "lap_dev":     compute_perplexity(dev_sentences, lap_fn),
        "lap_test":    compute_perplexity(test_sentences, lap_fn),
        "addk_dev":    compute_perplexity(dev_sentences, addk_fn),
        "addk_test":   compute_perplexity(test_sentences, addk_fn),
    }
    print(f"  {name} done.")

print("\nAll computations complete.")

Computing perplexity for MLE, Laplace (k=1), and Add-K (k=0.3)...

  Unigram done.
  Bigram done.
  Trigram done.
  Quadrigram done.

All computations complete.


In [15]:
# ── Three-way comparison table ───────────────────────────────────────────
def fmt(val):
    return f"{val:,.2f}" if not math.isinf(val) else "inf"

print("=" * 105)
print(f"COMPARISON: MLE vs. Laplace (k=1) vs. Add-K (k={K}) — Development Set Perplexity")
print("=" * 105)
print()
print(f"{'Model':<12} | {'MLE':>15} | {'Laplace (k=1)':>15} | {'Add-K (k=0.3)':>15}")
print("-" * 12 + "-+-" + "-" * 15 + "-+-" + "-" * 15 + "-+-" + "-" * 15)
for name in model_names:
    r = all_results[name]
    print(f"{name:<12} | {fmt(r['mle_dev']):>15} | {fmt(r['lap_dev']):>15} | {fmt(r['addk_dev']):>15}")

print()
print("=" * 105)
print(f"COMPARISON: MLE vs. Laplace (k=1) vs. Add-K (k={K}) — Test Set Perplexity")
print("=" * 105)
print()
print(f"{'Model':<12} | {'MLE':>15} | {'Laplace (k=1)':>15} | {'Add-K (k=0.3)':>15}")
print("-" * 12 + "-+-" + "-" * 15 + "-+-" + "-" * 15 + "-+-" + "-" * 15)
for name in model_names:
    r = all_results[name]
    print(f"{name:<12} | {fmt(r['mle_test']):>15} | {fmt(r['lap_test']):>15} | {fmt(r['addk_test']):>15}")

COMPARISON: MLE vs. Laplace (k=1) vs. Add-K (k=0.3) — Development Set Perplexity

Model        |             MLE |   Laplace (k=1) |   Add-K (k=0.3)
-------------+-----------------+-----------------+----------------
Unigram      |             inf |        6,384.81 |        6,507.09
Bigram       |             inf |       30,945.36 |       17,685.77
Trigram      |             inf |      167,963.14 |      121,630.75
Quadrigram   |             inf |      276,059.03 |      228,002.44

COMPARISON: MLE vs. Laplace (k=1) vs. Add-K (k=0.3) — Test Set Perplexity

Model        |             MLE |   Laplace (k=1) |   Add-K (k=0.3)
-------------+-----------------+-----------------+----------------
Unigram      |             inf |        6,386.94 |        6,547.45
Bigram       |             inf |       32,270.31 |       18,637.74
Trigram      |             inf |      173,218.43 |      126,452.18
Quadrigram   |             inf |      284,090.22 |      236,956.11


---
## 12. Summary

In [ ]:
print("=" * 80)
print(f"SUMMARY: Add-K Smoothing (K = {K})")
print("=" * 80)
print(f"\nCorpus           : IndicCorpV2 Telugu (tel_Telu)")
print(f"Total sentences  : {NUM_SENTENCES:,}")
print(f"Training set     : {TRAIN_SIZE:,} sentences")
print(f"Development set  : {DEV_SIZE:,} sentences")
print(f"Test set         : {TEST_SIZE:,} sentences")
print(f"Vocabulary size  : {V:,}")
print(f"K value          : {K}")
print()
print("Formula:")
print("  P_add-k(w_i | context) = (C(context, w_i) + k) / (C(context) + k * V)")
print()
print(f"{'Model':<12} {'Dev PP (Add-K)':>18} {'Test PP (Add-K)':>18}")
print("-" * 52)
for name in model_names:
    r = all_results[name]
    print(f"{name:<12} {fmt(r['addk_dev']):>18} {fmt(r['addk_test']):>18}")

print()
print("Key observations:") 
print("  - Add-K (k=0.3) is less aggressive than Laplace (k=1), preserving more")
print("    probability mass for observed n-grams.")
print("  - This typically results in LOWER perplexity than Laplace smoothing,")
print("    indicating a better fit to the held-out data.")
print("  - Both smoothing methods eliminate the infinite perplexity problem of MLE")
print("    by assigning non-zero probability to all possible n-grams.")
print("\n" + "=" * 80)

SUMMARY: Add-K Smoothing (K = 0.3)

Corpus           : IndicCorpV2 Telugu (tel_Telu)
Total sentences  : 1,000,000
Training set     : 998,000 sentences
Development set  : 1,000 sentences
Test set         : 1,000 sentences
Vocabulary size  : 670,753
K value          : 0.3

Formula:
  P_add-k(w_i | context) = (C(context, w_i) + k) / (C(context) + k * V)

Model            Dev PP (Add-K)    Test PP (Add-K)
----------------------------------------------------
Unigram                6,507.09           6,547.45
Bigram                17,685.77          18,637.74
Trigram              121,630.75         126,452.18
Quadrigram           228,002.44         236,956.11

Key observations:
  - Add-K (k=0.3) is less aggressive than Laplace (k=1), preserving more
    probability mass for observed n-grams.
  - This typically results in LOWER perplexity than Laplace smoothing,
    indicating a better fit to the held-out data.
  - Both smoothing methods eliminate the infinite perplexity problem of MLE
    by